# ProGen2 scoring for decoding-design-bias

Autoregressive decoder-only protein language model (Nijkamp et al., 2023).

Sequence-only — uses the `sequence` field of the dataset CSV (the same
input as ESM2). No PDB / structure dependency.

Scoring convention (matches the existing pipeline):
- For each WT sequence `s = s_1 ... s_L`, compute the total log-likelihood
  `sum_i log P(s_i | s_{<i})` from a single teacher-forced forward pass
  over the full sequence (shifted-logits trick).
- The reported `progen2_score` is the per-residue mean log-likelihood
  (`total / L`), so it is on the same scale as `proteinmpnn_score`,
  `esmif_score`, `ESM2_15B_pppl_score`, etc. — higher (closer to 0) means
  the model assigns more probability to the WT.
- The total log-likelihood and token count are also written for
  transparency, in case the methods section wants to renormalise.

ProGen2 tokenises per-residue (each amino acid is one token, plus
special tokens). This is the standard autoregressive scoring path.

Sequence-cleaning policy (matches the ESM2 script in
`Calculate_All_models_likelihoods_ORIGINAL.ipynb`):
- 20 standard amino acids only.
- Terminal `*` stripped if present; no replacement of X / B / Z / U / O.
- Proteins containing non-standard residues are recorded with NaN
  scores and `sequence_filter_status="nonstandard_amino_acid"`; the
  cohort actually scored is therefore comparable to ESM2's cohort.

Default checkpoint: `hugohrban/progen2-base`. Override via
`PROGEN2_CHECKPOINT` in the config cell to run a different size.

## 1. Setup

In [ ]:
!nvidia-smi -L


In [ ]:
# transformers + tokenizers are the only hard deps. Pin transformers to a
# version known to work with the ProGen2 architecture upload.
!pip install -q 'transformers>=4.40,<5' 'tokenizers>=0.15' 'accelerate>=0.30' biopython tqdm


In [ ]:
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/LBDillon/decoding-design-bias.git'
REPO_DIR = '/content/decoding-design-bias'
DATASET  = '/content/main_plus_r2_r3_scored_filterC_v4.csv'  # edit if needed

DRIVE_OUT_DIR = '/content/drive/MyDrive/decoding-design-bias/outputs'
OUTPUT        = f'{DRIVE_OUT_DIR}/progen2_scores.csv'
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)

# Edit to switch checkpoint. Brief defaults to progen2-base.
PROGEN2_CHECKPOINT = 'hugohrban/progen2-base'
MAX_TOKENS_PER_FORWARD = 2048  # halved on OOM

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=False)

assert os.path.exists(DATASET), DATASET
print('output ->', OUTPUT)
print('checkpoint ->', PROGEN2_CHECKPOINT)


## 2. Load model

In [ ]:
import torch
import warnings; warnings.filterwarnings('ignore')
from transformers import AutoModelForCausalLM, AutoTokenizer

assert torch.cuda.is_available(), 'Use a GPU runtime (Runtime -> Change runtime type -> GPU).'
device = torch.device('cuda')

tokenizer = AutoTokenizer.from_pretrained(PROGEN2_CHECKPOINT, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    PROGEN2_CHECKPOINT,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
).to(device).eval()

# ProGen2 uses '1' as BOS in the published code. Verify from the loaded tokeniser
# and fall back to '1' with a warning if the tokeniser doesn't expose it.
BOS_ID = tokenizer.bos_token_id
if BOS_ID is None:
    BOS_ID = 1
    print(f'tokenizer.bos_token_id is None; falling back to BOS_ID={BOS_ID} '
          f'(ProGen2 paper / Nijkamp code uses 1).')
else:
    print(f'BOS_ID from tokenizer: {BOS_ID}')

print(f'model dtype: {next(model.parameters()).dtype}')
print(f'vocab size:  {tokenizer.vocab_size}')


## 3. Sequence cleaning + scoring wrapper

In [ ]:
import pandas as pd
import math

VALID_STANDARD_AA = set('ACDEFGHIKLMNPQRSTVWY')

def clean_sequence(seq):
    """Match the ESM2 cleaning policy in the master notebook."""
    if seq is None or (isinstance(seq, float) and math.isnan(seq)):
        return '', 'missing'
    s = str(seq).strip().upper().replace(' ', '').replace('\n', '').replace('\r', '')
    if s.endswith('*'):
        s = s[:-1]
    bad_chars = ''.join(sorted(set(s) - VALID_STANDARD_AA))
    return s, bad_chars


def classify(seq_clean, bad_chars):
    if not seq_clean:
        return 'empty_sequence'
    if bad_chars:
        return 'nonstandard_amino_acid'
    return 'included'


@torch.no_grad()
def score_sequence_progen2(seq):
    """Teacher-forced total + mean log-likelihood for one WT sequence.

    Input tokens:   [BOS, t_1, ..., t_L]
    Forward pass:   logits[i] predicts t_{i+1} given t_{<=i}
    Score sum:      sum_{i=0..L-1} log p(t_{i+1} | t_{<=i})

    Returns a dict with
      - progen2_total_logp : sum of log p over all L positions (== full sequence LL)
      - progen2_score      : mean per-residue log p (= total / L)
      - scored_length      : L (number of residues actually scored)
      - num_tokens         : L (ProGen2 is one token per residue, so same as L)
    """
    enc = tokenizer(seq, add_special_tokens=False, return_tensors='pt')
    token_ids = enc['input_ids'][0]
    if BOS_ID is not None:
        token_ids = torch.cat([torch.tensor([BOS_ID]), token_ids])
    token_ids = token_ids.to(device)

    # Single forward pass; shift logits/labels to align next-token predictions.
    out = model(input_ids=token_ids.unsqueeze(0))
    logits = out.logits[0]                  # (T, V)
    log_probs = torch.log_softmax(logits.float(), dim=-1)

    # Predict token at index i+1 from logits at index i; labels are token_ids[1:]
    next_tokens = token_ids[1:]             # (T-1,)
    pred_log_probs = log_probs[:-1]         # (T-1, V)
    picked = pred_log_probs[torch.arange(next_tokens.shape[0], device=device), next_tokens]

    total_logp = float(picked.sum().item())
    L = int(next_tokens.shape[0])
    return {
        'progen2_total_logp': total_logp,
        'progen2_score':      total_logp / L if L else float('nan'),
        'scored_length':      L,
        'num_tokens':         L,
    }


def score_with_oom_retry(seq, max_tokens=MAX_TOKENS_PER_FORWARD):
    """Wrap the score call so a single OOM doesn't kill the run."""
    if len(seq) + 1 > max_tokens:
        # Single sequence longer than the budget — try anyway, but flag.
        pass
    try:
        return score_sequence_progen2(seq), None
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        return None, f'out_of_memory: L={len(seq)}'
    except Exception as exc:
        return None, repr(exc)


## 4. Sanity check on 3 proteins

In [ ]:
import csv, time

with open(DATASET) as fh:
    rows = list(csv.DictReader(fh))
print('dataset:', len(rows), 'rows')

for r in rows[:3]:
    entry = r['Entry']
    seq, bad = clean_sequence(r.get('sequence', ''))
    status = classify(seq, bad)
    if status != 'included':
        print(entry, status, bad); continue
    t0 = time.time(); res, err = score_with_oom_retry(seq); t = time.time() - t0
    if res is None:
        print(entry, 'failed:', err); continue
    print(f'{entry} L={res["scored_length"]:4d} '
          f'progen2_score={res["progen2_score"]:.4f}  total={res["progen2_total_logp"]:.2f}  '
          f'({t:.1f}s)')


## 5. Full run with resume

In [ ]:
from tqdm.auto import tqdm

already = set()
if os.path.exists(OUTPUT):
    with open(OUTPUT) as fh:
        for row in csv.DictReader(fh):
            if row.get('Entry'):
                already.add(row['Entry'])
    print('resuming, already scored:', len(already))

todo = [r for r in rows if r['Entry'] not in already]
# Score shortest first so OOM-prone tail comes last.
todo.sort(key=lambda r: len(r.get('sequence', '')))
print('remaining:', len(todo))

open_mode = 'a' if already else 'w'
FIELDS = [
    'Entry', 'species', 'domain',
    'progen2_score', 'progen2_total_logp',
    'scored_length', 'num_tokens',
    'dataset_length', 'sequence_filter_status', 'error',
]

with open(OUTPUT, open_mode, newline='') as out:
    w = csv.DictWriter(out, fieldnames=FIELDS)
    if open_mode == 'w':
        w.writeheader()

    counts = {'ok': 0, 'skipped_nonstd': 0, 'oom': 0, 'error': 0, 'empty': 0}
    t_start = time.time()
    for r in tqdm(todo, desc='progen2'):
        entry = r['Entry']
        rec = {
            'Entry': entry,
            'species': r.get('species', ''),
            'domain':  r.get('domain', ''),
            'progen2_score': '',
            'progen2_total_logp': '',
            'scored_length': 0,
            'num_tokens': 0,
            'dataset_length': len(r.get('sequence', '')),
            'sequence_filter_status': '',
            'error': '',
        }
        seq, bad = clean_sequence(r.get('sequence', ''))
        status = classify(seq, bad)
        rec['sequence_filter_status'] = status
        if status != 'included':
            if status == 'empty_sequence':
                counts['empty'] += 1
            else:
                counts['skipped_nonstd'] += 1
                rec['error'] = f'bad_chars={bad}'
            w.writerow(rec); out.flush(); continue

        res, err = score_with_oom_retry(seq)
        if res is None:
            rec['error'] = err or 'unknown_error'
            if err and 'out_of_memory' in err:
                counts['oom'] += 1
            else:
                counts['error'] += 1
            w.writerow(rec); out.flush(); continue

        rec.update(res)
        w.writerow(rec); out.flush()
        counts['ok'] += 1
        if counts['ok'] % 50 == 0:
            torch.cuda.empty_cache()

print('done in', round(time.time() - t_start, 1), 's', counts)


## 6. Quick look

In [ ]:
import pandas as pd
df = pd.read_csv(OUTPUT)
print(df.shape)
print('non-null progen2_score:', df['progen2_score'].notna().sum())
df[['progen2_score', 'progen2_total_logp', 'scored_length']].describe()
